# 이슈 #38 구조환경지표 원자료 기반 전수검증

## 목적과 현재 범위

- `configs/structural_indicators_verification.yaml`을 기준으로 구조환경지표 21개의 가공값과 원자료 연결 상태를 점검한다.
- 현재 상세 검증이 완료된 지표는 `2-1.1. 보육시설 보급률`이다. 나머지 지표의 산식 검증은 후속 작업이다.
- 기존 가공값은 `어린이집 수 ÷ 0–4세 인구 × 100`과 전부 일치했으며, 명세상 올바른 산식은 `어린이집 정원 ÷ 0–4세 인구 × 100`이다.
- 이 노트북은 수정값을 메모리에 생성할 뿐 원본 또는 기존 가공 파일을 덮어쓰지 않는다.

## 1. 실행 환경과 검증 명세 확인

저장소 루트를 찾고 필수 라이브러리를 불러온 뒤, 검증 명세와 구조환경 원자료 경로가 존재하는지 확인한다. YAML의 필수 구역, 지표 수, 지표 ID 중복 여부도 검사한다. 정상 출력은 이슈 `#38`, 검증 대상 `21개`, 고유 ID `21개`이다.

In [1]:
import hashlib
import unicodedata

import openpyxl
import pandas as pd
import yaml

from pathlib import Path

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".git").exists()), None
)

if repo_root is None:
    raise FileNotFoundError("현재 실행 위치의 상위 경로에서 Git 저장소를 찾지 못했습니다.")

print(f"저장소 루트: {repo_root}")

저장소 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


In [2]:
config_path = repo_root / "configs" / "structural_indicators_verification.yaml"
raw_root = repo_root / "data" / "raw"

missing_paths = [path for path in (config_path, raw_root) if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "다음 입력 경로를 찾지 못했습니다:\n" + "\n".join(map(str, missing_paths))
    )

print(f"검증 명세: {config_path.relative_to(repo_root)}")
print(f"구조환경 원자료: {raw_root.relative_to(repo_root)}")

검증 명세: configs/structural_indicators_verification.yaml
구조환경 원자료: data/raw


In [3]:
with config_path.open(encoding="utf-8") as file:
    manifest = yaml.safe_load(file)

required_sections = {"manifest_version", "issue", "scope", "rules", "local", "drive", "indicators"}
missing_sections = required_sections - manifest.keys()
if missing_sections:
    raise KeyError(f"YAML 필수 구역이 없습니다: {sorted(missing_sections)}")

indicators = manifest["indicators"]
if not isinstance(indicators, list) or not all(
    isinstance(indicator, dict) for indicator in indicators
):
    raise TypeError("indicators는 지표 딕셔너리로 구성된 리스트여야 합니다.")

indicator_ids = [indicator.get("id") for indicator in indicators]
if len(indicators) != manifest["scope"]["completed_indicators_to_validate"]:
    raise ValueError("검증 대상 지표 수가 scope의 선언값과 다릅니다.")
if None in indicator_ids or len(indicator_ids) != len(set(indicator_ids)):
    raise ValueError("지표 ID가 없거나 중복됐습니다.")

print(
    f"명세 버전: {manifest['manifest_version']} | 이슈: #{manifest['issue']} | "
    f"검증 대상: {len(indicators)}개 | 고유 ID: {len(set(indicator_ids))}개"
)

명세 버전: 3 | 이슈: #38 | 검증 대상: 21개 | 고유 ID: 21개


## 2. 명세와 로컬 원자료 연결 확인

YAML에 기록된 가공 파일·원자료 파일명을 로컬 원자료 폴더와 연결한다. 같은 이름의 파일이 여러 개면 SHA-256 해시로 실제 내용까지 비교한다. 현재 누락 파일은 없지만, 주민등록인구 파일은 같은 이름의 서로 다른 내용이 있어 지표에 맞는 파일을 명시적으로 선택해야 한다.

In [4]:
import re


def normalize_name(name: str) -> str:
    return unicodedata.normalize("NFC", name)


def core_key(name: str) -> str:
    # 파일명 자체가 아니라 "언제 재다운로드했는지"만 다른 경우를 찾기 위한 느슨한 키.
    # 8자리 날짜(예: 20181231)는 스냅샷 연도를 구분하는 의미가 있어 건드리지 않고,
    # 14자리 다운로드 시각(예: 20260715191848)과 맨 앞 연도범위·괄호주석만 제거한다.
    key = normalize_name(name)
    key = re.sub(r"^\([^)]*\)_", "", key)
    key = re.sub(r"^\d{4}(-\d{4})?_", "", key)
    key = re.sub(r"_\d{14}(?=\.[^.]+$)", "", key)
    return key


def extension_variants(name: str) -> set[str]:
    # 드라이브가 CSV를 엑셀로 다시 저장하면서 이름이 "name.csv.xlsx"로 늘어나거나
    # ".csv"가 통째로 ".xlsx"로 바뀌는 경우가 있어, 두 변형도 같이 찾아본다.
    variants = {name}
    if name.lower().endswith(".csv"):
        variants.add(name + ".xlsx")
        variants.add(name[:-4] + ".xlsx")
    return variants


def find_matches(paths_by_name: dict[str, list[Path]], file_name: str) -> list[Path]:
    for candidate in extension_variants(normalize_name(file_name)):
        matched = paths_by_name.get(candidate)
        if matched:
            return matched
    return []


local_files: list[Path] = sorted(
    (path for path in raw_root.rglob("*") if path.is_file()),
    key=lambda path: path.as_posix().casefold(),
)

if not local_files:
    raise FileNotFoundError(f"원자료 폴더가 비어 있습니다: {raw_root}")

paths_by_name: dict[str, list[Path]] = {}
paths_by_core_key: dict[str, list[Path]] = {}

for path in local_files:
    paths_by_name.setdefault(normalize_name(path.name), []).append(path)
    paths_by_core_key.setdefault(core_key(path.name), []).append(path)

# 재정팀 산출값 파일 안에 원자료가 시트로 이미 들어있는지 확인한다.
# 시트가 2개 이상이면 계산 시트·원자료 시트가 포함된 것으로 본다.
# 보육시설 보급률(numerator_active, 정원 데이터)만 예외로, 파일 안에 없는 게 확인됐다.
EMBEDDED_ROLE_EXCEPTIONS = {("childcare_capacity_rate", "numerator_active")}

# YAML rules.no_separate_raw_data_indicators에 등록된, 애초에 별도 원자료가 없는 지표.
NO_RAW_DATA_INDICATORS = set(manifest["rules"].get("no_separate_raw_data_indicators", []))

sheet_counts_by_indicator: dict[str, int] = {}

for indicator in indicators:
    matched = find_matches(paths_by_name, indicator["finance_team_value"]["file_name"])
    if not matched:
        continue

    workbook = openpyxl.load_workbook(matched[0], read_only=True)
    sheet_counts_by_indicator[indicator["id"]] = len(workbook.sheetnames)
    workbook.close()

mapping_records: list[dict[str, str]] = []

for indicator in indicators:
    mapping_records.append(
        {
            "indicator_id": indicator["id"],
            "mapping_type": "finance_team_value",
            "role": "finance_team_value",
            "drive_id": indicator["finance_team_value"]["id"],
            "file_name": indicator["finance_team_value"]["file_name"],
            "embedded_in_finance_team_value": False,
            "no_raw_data_expected": False,
        }
    )

    for source in indicator["raw_sources"]:
        embedded = (
            sheet_counts_by_indicator.get(indicator["id"], 1) > 1
            and (indicator["id"], source["role"]) not in EMBEDDED_ROLE_EXCEPTIONS
        )
        mapping_records.append(
            {
                "indicator_id": indicator["id"],
                "mapping_type": "raw",
                "role": source["role"],
                "drive_id": source["id"],
                "file_name": source["file_name"],
                "embedded_in_finance_team_value": embedded,
                "no_raw_data_expected": indicator["id"] in NO_RAW_DATA_INDICATORS,
            }
        )

resolution_rows: list[dict[str, object]] = []

for record in mapping_records:
    matched_paths = find_matches(paths_by_name, record["file_name"])
    candidate_paths = (
        [] if matched_paths else paths_by_core_key.get(core_key(record["file_name"]), [])
    )

    resolution_rows.append(
        {
            **record,
            "match_count": len(matched_paths),
            "local_paths": " | ".join(str(path.relative_to(repo_root)) for path in matched_paths),
            "timestamp_candidate_count": len(candidate_paths),
            "timestamp_candidate_paths": " | ".join(
                str(path.relative_to(repo_root)) for path in candidate_paths
            ),
        }
    )

file_resolution = pd.DataFrame(resolution_rows)
file_resolution["embedded_in_finance_team_value"] = file_resolution[
    "embedded_in_finance_team_value"
].astype(bool)
file_resolution["no_raw_data_expected"] = file_resolution["no_raw_data_expected"].astype(bool)

still_needed = file_resolution[
    ~file_resolution["embedded_in_finance_team_value"] & ~file_resolution["no_raw_data_expected"]
].copy()

no_candidate = still_needed[
    (still_needed["match_count"] == 0) & (still_needed["timestamp_candidate_count"] == 0)
].copy()

timestamp_candidates = still_needed[
    (still_needed["match_count"] == 0) & (still_needed["timestamp_candidate_count"] > 0)
].copy()

ambiguous_file_mappings = still_needed[still_needed["match_count"] > 1].copy()

embedded_count = file_resolution["embedded_in_finance_team_value"].sum()
no_raw_data_count = file_resolution["no_raw_data_expected"].sum()

print(
    f"로컬 탐색 파일: {len(local_files)}개 | "
    f"매핑 참조: {len(file_resolution)}건 | "
    f"파일 안에 이미 있음(임베디드): {embedded_count}건 | "
    f"원자료 없음(정상): {no_raw_data_count}건 | "
    f"정확히 1개 일치: {(still_needed['match_count'] == 1).sum()}건 | "
    f"타임스탬프만 다른 후보: {len(timestamp_candidates)}건 | "
    f"진짜 누락(후보 없음): {len(no_candidate)}건 | "
    f"다중 일치: {len(ambiguous_file_mappings)}건"
)

with pd.option_context("display.max_colwidth", None):
    if not timestamp_candidates.empty:
        print(f"\n타임스탬프만 다른 후보 발견 - 확인 필요 ({len(timestamp_candidates)}건):")
        display(
            timestamp_candidates[["indicator_id", "role", "file_name", "timestamp_candidate_paths"]]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

    if not no_candidate.empty:
        print(
            f"\n진짜 누락된 매핑 ({len(no_candidate)}건, 임베디드·원자료없음·타임스탬프후보 제외):"
        )
        display(
            no_candidate[["indicator_id", "role", "file_name"]]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

    if not ambiguous_file_mappings.empty:
        print(f"\n다중 일치 매핑 ({len(ambiguous_file_mappings)}건):")
        display(
            ambiguous_file_mappings[
                ["indicator_id", "role", "file_name", "match_count", "local_paths"]
            ]
            .sort_values(["indicator_id", "role"])
            .reset_index(drop=True)
        )

로컬 탐색 파일: 67개 | 매핑 참조: 65건 | 파일 안에 이미 있음(임베디드): 37건 | 원자료 없음(정상): 4건 | 정확히 1개 일치: 24건 | 타임스탬프만 다른 후보: 0건 | 진짜 누락(후보 없음): 0건 | 다중 일치: 0건


In [5]:
def calculate_sha256(path: Path) -> str:
    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


ambiguous_names = sorted(
    str(file_name) for file_name in ambiguous_file_mappings["file_name"].unique()
)

if not ambiguous_names:
    print("다중 일치 파일명이 없어 SHA-256 비교를 건너뜁니다.")
else:
    hash_rows: list[dict[str, object]] = []

    for file_name in ambiguous_names:
        for path in paths_by_name[normalize_name(file_name)]:
            hash_rows.append(
                {
                    "file_name": file_name,
                    "local_path": path.relative_to(raw_root).as_posix(),
                    "size_bytes": path.stat().st_size,
                    "sha256": calculate_sha256(path),
                }
            )

    duplicate_file_hashes = pd.DataFrame(hash_rows)

    duplicate_content_summary = duplicate_file_hashes.groupby("file_name", as_index=False).agg(
        candidate_count=("local_path", "size"),
        distinct_size_count=("size_bytes", "nunique"),
        distinct_hash_count=("sha256", "nunique"),
    )

    duplicate_content_summary["content_identical"] = (
        duplicate_content_summary["distinct_hash_count"] == 1
    )

    different_content_files = duplicate_content_summary[
        ~duplicate_content_summary["content_identical"]
    ].copy()

    print(
        f"중복 파일명: {len(duplicate_content_summary)}개 | "
        f"후보 파일: {len(duplicate_file_hashes)}개 | "
        f"내용 동일: {duplicate_content_summary['content_identical'].sum()}개 | "
        f"내용 다름: {len(different_content_files)}개"
    )

    if not different_content_files.empty:
        print("\n내용이 다른 중복 파일:")
        display(different_content_files)
        display(
            duplicate_file_hashes[
                duplicate_file_hashes["file_name"].isin(different_content_files["file_name"])
            ]
        )

다중 일치 파일명이 없어 SHA-256 비교를 건너뜁니다.


## 3. 보육시설 보급률 입력 파일 확정

- `finance_team_value`: 검증할 기존 보육시설 보급률
- `numerator_active`: 올바른 분자인 어린이집 정원 (아직 로컬에 없음 - 별도로 받아야 함)
- `numerator_excluded_facility_count`: 기존 오류 산식을 입증하기 위한 어린이집 수 비교자료이며, 수정 산식의 분자로 사용하지 않는다.

분모(0–4세 주민등록인구)는 별도 파일이 아니라 `finance_team_value`(`2-1.1. 보육시설 보급률.xlsx`)의 "0-4세 인구" 시트에 이미 포함돼 있어, 여기서는 따로 찾지 않는다.

In [6]:
# cell 9

childcare = next(item for item in indicators if item["id"] == "childcare_capacity_rate")

EMBEDDED_ROLES = {"denominator", "numerator_excluded_facility_count"}

childcare_paths = {
    "finance_team_value": paths_by_name[childcare["finance_team_value"]["file_name"]][0],
    **{
        source["role"]: paths_by_name[source["file_name"]][0]
        for source in childcare["raw_sources"]
        if source["role"] not in EMBEDDED_ROLES
    },
}

for role, path in childcare_paths.items():
    print(f"{role}: {path.relative_to(repo_root)}")

finance_team_value: data/raw/지표별_측정값/2-1.1. 보육시설 보급률.xlsx
numerator_active: data/raw/지표별_원데이터/2-1.1. 보육시설 보급률 원자료/전국_어린이집_정현원_현황_20260724113247.csv.xlsx


## 4. 입력 자료 구조 확인

선택한 엑셀 파일의 시트명, 행·열 크기, 상단 값을 미리 확인한다. 이 단계에서는 값을 변환하지 않으며, 2016–2025년 지역별 정원과 0–4세 인구가 실제로 들어 있는지 확인한다.

In [7]:
# cell 10

for role, path in childcare_paths.items():
    if path.suffix.lower() != ".xlsx":
        continue

    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    print(f"\n[{role}] {path.name}")

    for worksheet in workbook.worksheets:
        preview = list(worksheet.iter_rows(max_row=8, values_only=True))
        print(f"{worksheet.title}: {worksheet.max_row}행 × {worksheet.max_column}열")
        display(pd.DataFrame(preview))

    workbook.close()


[finance_team_value] 2-1.1. 보육시설 보급률.xlsx
2-1.1. 보육시설 보급률: 19행 × 12열


,0,1,2,3,4,5,6,7,8,9,10,11
0,지역,세부지표,2016.000000,2017.000000,2018.000000,2019.000000,2020.000000,2021.000000,2022.000000,2023.000000,2024.000000,2025.000000
1,전국,보육시설 보급률,1.863836,1.935343,1.984101,2.025395,2.108021,2.172275,2.166098,2.169508,2.153445,2.077639
2,서울,보육시설 보급률,1.691031,1.777450,1.851635,1.894288,1.985352,2.064710,2.070126,2.080145,2.075930,1.979006
3,부산,보육시설 보급률,1.466935,1.551139,1.614956,1.711698,1.825949,1.900248,1.903485,1.920117,1.903396,1.820668
4,대구,보육시설 보급률,1.494990,1.561200,1.576827,1.596483,1.708895,1.788809,1.860442,1.873687,1.886139,1.819249
5,인천,보육시설 보급률,1.711152,1.793656,1.855752,1.905036,2.017234,2.031458,1.983658,1.996302,1.981792,1.875218
6,광주,보육시설 보급률,1.893980,2.023730,2.061376,2.080398,2.198073,2.209531,2.211817,2.256141,2.294264,2.267541
7,대전,보육시설 보급률,2.322172,2.381291,2.396210,2.410901,2.485110,2.532723,2.434335,2.305505,2.258587,2.167126


보육시설 보급률 계산: 57행 × 13열


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,지역,항목,NaN,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
1,전국,총인구수[명],0-4세,2204271,2079115,1974244,1845122,1677023,1530469,1427590,1334588,1271776,1254501
2,서울,총인구수[명],0-4세,376575,350277,324470,300799,270481,244538,227619,213014,202897,202627
3,부산,총인구수[명],0-4세,132044,123780,117093,107963,97374,87778,81272,75360,71136,70194
4,대구,총인구수[명],0-4세,99198,93774,89103,82807,74200,66357,61222,57587,54874,54528
5,인천,총인구수[명],0-4세,130380,121874,115371,107557,96320,88754,85549,82753,81391,82977
6,광주,총인구수[명],0-4세,65365,61273,57971,53932,48770,45349,42499,38916,36090,34619
7,대전,총인구수[명],0-4세,68212,63201,58676,53424,47684,43471,41613,39731,38254,38115


0-4세 인구: 109행 × 13열


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,지역,연령별,항목,2016 년,2017 년,2018 년,2019 년,2020 년,2021 년,2022 년,2023 년,2024 년,2025 년
1,전국,0세,총인구수[명],393674,345786,317685,295132,265087,253946,244250,225958,235337,252212
2,전국,1세,총인구수[명],441720,409814,361625,330970,304651,274633,264788,253595,234405,243765
3,전국,2세,총인구수[명],439207,442943,411225,362900,331606,306120,277529,266619,254654,235297
4,전국,3세,총인구수[명],440530,439700,443586,412018,363250,332157,307975,279134,267456,255233
5,전국,4세,총인구수[명],489140,440872,440123,444102,412429,363613,333048,309282,279924,267994
6,서울,0세,총인구수[명],70798,61253,54719,51145,45165,43410,40742,37771,39588,43645
7,서울,1세,총인구수[명],76955,70532,60805,54779,50482,44904,43950,41087,37696,39703


어린이집 현황: 19행 × 11열


,0,1,2,3,4,5,6,7,8,9,10
0,시도별,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1,전국,41084,40238,39171,37371,35352,33246,30923,28954,27387,26064
2,서울특별시,6368,6226,6008,5698,5370,5049,4712,4431,4212,4010
3,부산광역시,1937,1920,1891,1848,1778,1668,1547,1447,1354,1278
4,대구광역시,1483,1464,1405,1322,1268,1187,1139,1079,1035,992
5,인천광역시,2231,2186,2141,2049,1943,1803,1697,1652,1613,1556
6,광주광역시,1238,1240,1195,1122,1072,1002,940,878,828,785
7,대전광역시,1584,1505,1406,1288,1185,1101,1013,916,864,826



[numerator_active] 전국_어린이집_정현원_현황_20260724113247.csv.xlsx
전국_어린이집_정현원_현황_20260724113247: 19행 × 12열


,0,1,2,3,4,5,6,7,8,9,10,11
0,시도별,항목,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
1,전국,정원,1767224,1756603,1732324,1686873,1628260,1557496,1476686,1401440,1340087,1285032
2,서울특별시,정원,270231,268100,263157,254538,245863,235682,223628,211248,202722,195742
3,부산광역시,정원,89658,89040,88222,86553,84008,80248,76146,71570,66235,62266
4,대구광역시,정원,75255,74696,72500,69390,66727,62125,59434,56462,53946,52133
5,인천광역시,정원,94314,93938,93888,91615,88002,84877,80896,78883,76831,75036
6,광주광역시,정원,62528,63161,61121,57756,55020,49556,46531,43399,41021,38868
7,대전광역시,정원,54514,53688,51025,47923,45160,42171,39145,36356,34983,34111


### 비교용 어린이집 수 자료

기존 가공값이 어린이집 수를 사용했는지 확인한다. 이 자료도 별도 파일이 아니라 `finance_team_value`의 "어린이집 현황" 시트에 이미 들어있어, 여기서 바로 읽는다.

In [ ]:
# cell 11

facility_count_raw = pd.read_excel(
    childcare_paths["finance_team_value"], sheet_name="어린이집 현황"
)

print(f"크기: {facility_count_raw.shape[0]}행 × {facility_count_raw.shape[1]}열")
print(f"열 이름: {facility_count_raw.columns.tolist()}")

display(facility_count_raw.head(8))

## 5. 정원·시설 수·0–4세 인구 결합

지역명을 `서울특별시 → 서울`처럼 통일하고 세 자료를 `지역 × 연도` 긴 형식으로 변환한다. 0–4세 인구는 별도 파일이 아니라 `finance_team_value`의 "0-4세 인구" 시트에서 읽어 `총인구수[명]`만 합산한다. 이후 세 자료를 일대일 결합하고, 18개 지역 × 10개 연도인 180행과 결측값 없음까지 검증한다. 결과 변수는 `childcare_comparison`이다.

In [ ]:
# cell 12

years = [str(y) for y in range(2016, 2026)]
region_order = [
    "전국",
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주",
]

region_map = {r: r for r in region_order} | {
    "서울특별시": "서울",
    "부산광역시": "부산",
    "대구광역시": "대구",
    "인천광역시": "인천",
    "광주광역시": "광주",
    "대전광역시": "대전",
    "울산광역시": "울산",
    "세종특별자치시": "세종",
    "경기도": "경기",
    "강원도": "강원",
    "강원특별자치도": "강원",
    "충청북도": "충북",
    "충청남도": "충남",
    "전라북도": "전북",
    "전북특별자치도": "전북",
    "전라남도": "전남",
    "경상북도": "경북",
    "경상남도": "경남",
    "제주특별자치도": "제주",
}


def wide_to_long(df, region_col, value_col):
    df = df.copy()
    df.columns = df.columns.map(lambda x: str(x).strip())
    df["지역"] = df[region_col].astype(str).str.strip().map(region_map)

    if df["지역"].isna().any():
        raise ValueError(f"{value_col}: 미등록 지역명이 있습니다.")
    if set(df["지역"]) != set(region_order) or df["지역"].duplicated().any():
        raise ValueError(f"{value_col}: 지역 누락 또는 중복이 있습니다.")

    result = df[["지역", *years]].melt(id_vars="지역", var_name="연도", value_name=value_col)
    result["연도"] = result["연도"].astype(int)
    result[value_col] = pd.to_numeric(result[value_col], errors="raise")
    return result


capacity_raw = pd.read_excel(childcare_paths["numerator_active"])

population_raw = pd.read_excel(childcare_paths["finance_team_value"], sheet_name="0-4세 인구")
population_raw.columns = population_raw.columns.map(lambda x: str(x).strip())
population_raw = population_raw.rename(columns={f"{y} 년": y for y in years})

for col in ["지역", "연령별", "항목"]:
    population_raw[col] = population_raw[col].astype(str).str.strip()

population_selected = population_raw[
    population_raw["지역"].isin(region_map)
    & population_raw["연령별"].isin([f"{age}세" for age in range(5)])
    & population_raw["항목"].eq("총인구수[명]")
].copy()

population_selected["지역"] = population_selected["지역"].map(region_map)

age_counts = population_selected.groupby("지역").size().reindex(region_order, fill_value=0)
if not age_counts.eq(5).all():
    raise ValueError(f"지역별 0–4세 자료 오류:\n{age_counts[age_counts.ne(5)]}")

population_selected[years] = population_selected[years].apply(pd.to_numeric, errors="raise")
population_wide = population_selected.groupby("지역", as_index=False)[years].sum()

childcare_comparison = (
    wide_to_long(capacity_raw, "시도별", "어린이집_정원")
    .merge(
        wide_to_long(facility_count_raw, "시도별", "어린이집_수"),
        on=["지역", "연도"],
        validate="one_to_one",
    )
    .merge(
        wide_to_long(population_wide, "지역", "0_4세_인구"),
        on=["지역", "연도"],
        validate="one_to_one",
    )
)

order_map = {r: i for i, r in enumerate(region_order)}
childcare_comparison["_순서"] = childcare_comparison["지역"].map(order_map)
childcare_comparison = (
    childcare_comparison.sort_values(["_순서", "연도"]).drop(columns="_순서").reset_index(drop=True)
)

if len(childcare_comparison) != 180 or childcare_comparison.isna().any().any():
    raise ValueError("결합 결과에 행 수 오류 또는 결측값이 있습니다.")

print(f"결합 결과: {childcare_comparison.shape}")
display(childcare_comparison.head(12))

## 6. 기존 산식 오류 검증

두 산식을 같은 지역·연도 자료로 계산한다.

- 정원 기준: `어린이집 정원 ÷ 0–4세 인구 × 100`
- 개수 기준: `어린이집 수 ÷ 0–4세 인구 × 100`

기존 가공값과 개수 기준 값의 절대오차를 계산한다. `기존값 최대 오차: 0.0`은 180개 지역·연도 값 모두가 개수 기준 산식과 정확히 일치한다는 뜻이며, 기존 산식 오류가 확정된다.

In [ ]:
# cell 13

childcare_comparison["정원_기준_보급률"] = (
    childcare_comparison["어린이집_정원"] / childcare_comparison["0_4세_인구"] * 100
)
childcare_comparison["개수_기준_보급률"] = (
    childcare_comparison["어린이집_수"] / childcare_comparison["0_4세_인구"] * 100
)

existing_raw = pd.read_excel(
    childcare_paths["finance_team_value"], sheet_name="2-1.1. 보육시설 보급률"
)
existing_rate = wide_to_long(existing_raw, "지역", "기존_보급률")

childcare_comparison = childcare_comparison.merge(
    existing_rate, on=["지역", "연도"], validate="one_to_one"
)
childcare_comparison["기존값_오차"] = (
    childcare_comparison["기존_보급률"] - childcare_comparison["개수_기준_보급률"]
).abs()

print("기존값 최대 오차:", childcare_comparison["기존값_오차"].max())
display(childcare_comparison.head(12).round(6))

## 7. 수정값 표 생성과 인계 상태

`정원_기준_보급률`을 기존 가공 파일과 같은 `지역 × 연도` 형태로 변환해 `childcare_corrected`에 저장한다. 정상 결과는 18행 × 11열이며 결측값이 없어야 한다.

**현재 완료:** 보육시설 보급률의 원자료 연결, 기존 오류 입증, 2016–2025년 수정값 생성.

**후속 작업:** 수정값을 실제 가공 파일·통합본에 반영하고 일치 여부를 재검증한 뒤, 나머지 구조환경지표의 산식 검증을 이어간다.

In [ ]:
# cell 14: 수정된 보육시설 보급률 표 생성

childcare_corrected = (
    childcare_comparison.pivot(index="지역", columns="연도", values="정원_기준_보급률")
    .reindex(index=region_order, columns=range(2016, 2026))
    .reset_index()
)

if childcare_corrected.shape != (18, 11) or childcare_corrected.isna().any().any():
    raise ValueError("수정 결과의 지역·연도 또는 결측값을 확인하세요.")

display(childcare_corrected.round(6))